# 01 — Compliance auditor architecture

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/04_compliance_auditor/01_architecture.ipynb)

**Prerequisites**:
- [00_first_principles.ipynb](./00_first_principles.ipynb)

**What you will learn**:
- End-to-end architecture for an automated compliance auditor
- How to extract structured requirements from regulation text
- Policy corpus chunking that preserves section hierarchy
- Semantic coverage assessment via retrieval + LLM scoring
- Gap analysis aggregation and remediation generation
- Change-monitoring pipeline for evolving regulations

In [ ]:
# @title Setup — Run this cell first
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "chromadb>=0.4.0"

import os, json, re, textwrap
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple
import numpy as np

# ── OpenAI client (optional — used for LLM scoring) ──
from openai import OpenAI

client: Optional[OpenAI] = None
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM calls will use mock responses")

print("Setup complete ✓")

## Section 1 — System architecture

```
┌────────────────────────────────────────────────────────────────┐
│                  Automated Compliance Auditor                  │
├────────────────────────────────────────────────────────────────┤
│                                                                │
│  Regulation Text ──► Requirement Extractor                     │
│                          │                                     │
│                          ▼                                     │
│                  Structured Requirements DB                    │
│                          │                                     │
│  Policy Corpus ──► Policy Retriever ◄──── Sentence Embeddings │
│                          │                                     │
│                          ▼                                     │
│                  Coverage Assessor (LLM)                       │
│                          │                                     │
│                          ▼                                     │
│                  Gap Reporter                                  │
│                          │                                     │
│                          ▼                                     │
│                  Remediation Advisor (LLM)                     │
│                          │                                     │
│  Regulation Monitor ─────┘  (periodic diff + re-assess)       │
│                                                                │
└────────────────────────────────────────────────────────────────┘
```

Each component is a well-defined stage. This notebook designs *schemas and contracts*
between them; Notebook 02 implements the full pipeline.

In [ ]:
# ── Pipeline stage registry ──
@dataclass
class PipelineStage:
    """Metadata for a pipeline stage."""
    name: str
    input_type: str
    output_type: str
    description: str

stages = [
    PipelineStage("Requirement Extractor", "str (regulation text)", "List[Requirement]",
                  "Parse regulation into structured, hierarchical requirements."),
    PipelineStage("Policy Retriever", "str (requirement text)", "List[PolicyChunk]",
                  "Retrieve top-k policy sections semantically matching the requirement."),
    PipelineStage("Coverage Assessor", "Requirement + List[PolicyChunk]", "CoverageResult",
                  "Score 1-5 how well retrieved policies satisfy the requirement."),
    PipelineStage("Gap Reporter", "List[CoverageResult]", "GapReport",
                  "Aggregate scores, identify critical gaps, rank by severity."),
    PipelineStage("Remediation Advisor", "GapReport", "List[Remediation]",
                  "Generate specific policy language and actions to close each gap."),
    PipelineStage("Regulation Monitor", "str (old) + str (new)", "List[Change]",
                  "Diff regulation versions, classify changes, trigger re-assessment."),
]

print("Pipeline stages")
print("=" * 80)
for i, s in enumerate(stages, 1):
    print(f"\n{i}. {s.name}")
    print(f"   Input  : {s.input_type}")
    print(f"   Output : {s.output_type}")
    print(f"   {s.description}")

## Section 2 — Requirement extraction

A requirement extractor must produce a **structured schema** from free-form
regulation text. Every extracted item carries:

| Field | Type | Description |
|-------|------|-------------|
| `id` | str | Unique identifier (e.g. `GDPR-13-1-a`) |
| `text` | str | Full requirement text |
| `article` | str | Source article reference |
| `category` | str | Thematic category (consent, security, rights, …) |
| `obligation_type` | str | `must` / `should` / `may` |
| `conditions` | list | Pre-conditions for applicability |
| `parent` | str | Parent requirement id (for hierarchy) |

In [ ]:
@dataclass
class ExtractedRequirement:
    """Structured requirement extracted from regulation text."""
    id: str
    text: str
    article: str
    category: str
    obligation_type: str          # must | should | may
    conditions: List[str] = field(default_factory=list)
    parent: Optional[str] = None

def extract_requirements(regulation_text: str) -> List[ExtractedRequirement]:
    """Rule-based extractor for GDPR-style articles.

    Uses regex patterns to identify article numbers, obligation keywords,
    and nested sub-paragraphs. Production systems would combine this with
    an LLM for better coverage.
    """
    requirements: List[ExtractedRequirement] = []
    article_pat = re.compile(r"Article\s+(\d+)(?:\((\d+)\))?")
    obligation_map = {"shall": "must", "must": "must", "should": "should", "may": "may"}

    # Split into paragraphs
    paragraphs = [p.strip() for p in regulation_text.split("\n") if p.strip()]

    current_article = ""
    parent_id = None
    for para in paragraphs:
        # Check for article header
        art_match = article_pat.search(para)
        if art_match:
            current_article = f"Art. {art_match.group(1)}"
            parent_id = f"GDPR-{art_match.group(1)}"

        # Detect obligation type
        obl = "must"
        for keyword, obl_type in obligation_map.items():
            if keyword in para.lower():
                obl = obl_type
                break

        # Detect sub-paragraph (a), (b), …
        sub_match = re.match(r"^\(([a-z])\)\s+(.+)", para)
        if sub_match:
            sub_letter = sub_match.group(1)
            text = sub_match.group(2)
            req_id = f"{parent_id}-{sub_letter}" if parent_id else f"REQ-{sub_letter}"
            requirements.append(ExtractedRequirement(
                id=req_id, text=text, article=current_article,
                category="general", obligation_type=obl,
                parent=parent_id
            ))
        elif len(para) > 40 and any(k in para.lower() for k in obligation_map):
            req_id = parent_id or f"REQ-{len(requirements)+1}"
            requirements.append(ExtractedRequirement(
                id=req_id, text=para, article=current_article,
                category="general", obligation_type=obl
            ))

    return requirements

# ── Test on sample GDPR articles ──
sample_regulation = """
Article 13(1) — Information to be provided where personal data are collected
The controller shall, at the time when personal data are obtained, provide the data subject with:
(a) the identity and the contact details of the controller
(b) the contact details of the data protection officer
(c) the purposes of the processing and the legal basis
(d) the legitimate interests pursued by the controller or third party
(e) the recipients or categories of recipients of the personal data
(f) the intention to transfer personal data to a third country

Article 15(1) — Right of access by the data subject
The data subject shall have the right to obtain from the controller confirmation as to whether personal data concerning him or her are being processed.

Article 17(1) — Right to erasure
The data subject shall have the right to obtain from the controller the erasure of personal data concerning him or her without undue delay where one of the following grounds applies:
(a) the personal data are no longer necessary for the purposes for which they were collected
(b) the data subject withdraws consent and there is no other legal ground
(c) the data subject objects pursuant to Article 21(1)
(d) the personal data have been unlawfully processed
(e) the personal data have to be erased for compliance with a legal obligation
"""

reqs = extract_requirements(sample_regulation)
print(f"Extracted {len(reqs)} requirements from sample text\n")
for r in reqs:
    parent_str = f" (child of {r.parent})" if r.parent else ""
    print(f"  {r.id:15s} [{r.obligation_type:5s}] {r.text[:65]}…{parent_str}")

## Section 3 — Policy corpus management

Company policies must be **chunked** intelligently for retrieval:

- Preserve section headers so context is retained after chunking
- Respect paragraph boundaries — never split mid-sentence
- Overlap slightly between chunks for continuity

We index chunks with **sentence-transformers** for semantic search.

In [ ]:
@dataclass
class PolicyChunk:
    """A chunk of corporate policy text with metadata."""
    id: str
    source: str       # e.g. "Privacy Policy"
    section: str      # e.g. "§4 Data Deletion"
    text: str
    embedding: Optional[List[float]] = field(default=None, repr=False)

def chunk_policy(name: str, sections: Dict[str, str],
                 max_tokens: int = 200) -> List[PolicyChunk]:
    """Chunk a policy document by section, splitting long sections."""
    chunks: List[PolicyChunk] = []
    for sec_title, sec_text in sections.items():
        words = sec_text.split()
        for i in range(0, len(words), max_tokens):
            chunk_words = words[i:i + max_tokens]
            chunk_text = f"[{name} — {sec_title}]\n" + " ".join(chunk_words)
            chunks.append(PolicyChunk(
                id=f"{name}::{sec_title}::chunk{i // max_tokens}",
                source=name,
                section=sec_title,
                text=chunk_text,
            ))
    return chunks

# ── Build a small corpus ──
privacy_policy = {
    "§1 Introduction": (
        "This privacy policy describes how Acme Corp collects, uses, and protects "
        "personal data of its users. We are committed to transparency and compliance "
        "with applicable data protection regulations."
    ),
    "§2 Data Collection": (
        "We collect personal data that you provide directly, such as name, email address, "
        "and payment information. We also collect usage data through cookies and analytics "
        "tools. The legal basis for processing is your consent and our legitimate interests."
    ),
    "§3 Data Access": (
        "Users may request a copy of all personal data we hold by contacting support. "
        "We will respond within 30 days. Data is provided in PDF format."
    ),
    "§4 Data Deletion": (
        "Users can request complete deletion of their account and associated data at any "
        "time through the account settings page. Deletion is processed within 72 hours. "
        "Backup copies are purged within 30 days."
    ),
}

retention_policy = {
    "§1 Retention Periods": (
        "Personal data is retained for the duration of the user relationship plus 12 months. "
        "Financial records are retained for 7 years as required by law. Anonymised analytics "
        "data may be retained indefinitely."
    ),
    "§2 Automated Deletion": (
        "Inactive accounts are flagged after 18 months and deleted after 24 months of "
        "inactivity. Users receive a notification 30 days before scheduled deletion."
    ),
}

security_policy = {
    "§1 Encryption": (
        "All data is encrypted at rest using AES-256 and in transit using TLS 1.3. "
        "Encryption keys are rotated quarterly and stored in a hardware security module."
    ),
    "§2 Access Control": (
        "Access to personal data is restricted to authorised personnel on a need-to-know "
        "basis. Multi-factor authentication is required for all internal systems. "
        "Access reviews are conducted quarterly."
    ),
    "§3 Incident Response": (
        "Security incidents are classified by severity. Critical incidents are escalated "
        "within 1 hour. Post-incident reviews are conducted within 48 hours."
    ),
}

all_chunks: List[PolicyChunk] = []
for name, sections in [("Privacy Policy", privacy_policy),
                        ("Data Retention Policy", retention_policy),
                        ("Security Policy", security_policy)]:
    all_chunks.extend(chunk_policy(name, sections))

print(f"Corpus: {len(all_chunks)} chunks from 3 policies\n")
for c in all_chunks:
    print(f"  {c.id:50s}  ({len(c.text.split())} words)")

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode all chunks
texts = [c.text for c in all_chunks]
embeddings = encoder.encode(texts, show_progress_bar=False)
for chunk, emb in zip(all_chunks, embeddings):
    chunk.embedding = emb.tolist()

def retrieve(query: str, top_k: int = 3) -> List[Tuple[PolicyChunk, float]]:
    """Retrieve top-k policy chunks by cosine similarity."""
    q_emb = encoder.encode([query])
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(all_chunks[i], float(sims[i])) for i in top_idx]

# Test retrieval
test_queries = [
    "right to erasure of personal data",
    "encryption and security measures",
    "data retention periods",
]
for q in test_queries:
    results = retrieve(q, top_k=2)
    print(f"\nQuery: '{q}'")
    for chunk, sim in results:
        print(f"  [{sim:.3f}] {chunk.id}")

## Section 4 — Semantic coverage assessment

For each requirement, the **Coverage Assessor** retrieves top-k policy chunks and
asks an LLM to score coverage on a 1-5 scale.

The assessment prompt must be carefully designed:
- Provide the requirement text and obligation type
- Provide the retrieved policy excerpts
- Ask for: score, evidence quote, gap description
- Enforce structured JSON output

In [ ]:
COVERAGE_PROMPT = """You are a compliance auditor. Assess how well the provided policy
excerpts satisfy the given regulatory requirement.

REQUIREMENT ({obligation_type}):
{requirement_text}

POLICY EXCERPTS:
{policy_excerpts}

Score the coverage on this scale:
5 = Fully covered — policy directly addresses requirement with specifics
4 = Substantially covered — most aspects addressed, minor gaps
3 = Partially covered — some relevant language, significant gaps
2 = Weakly addressed — tangentially related, does not satisfy
1 = Not addressed — no relevant policy language found

Respond in this exact JSON format:
{{
  "score": <int 1-5>,
  "evidence": "<exact quote from policy that best supports coverage>",
  "gap_description": "<what is missing or insufficient>",
  "confidence": <float 0-1>
}}"""

@dataclass
class CoverageResult:
    """Result of assessing one requirement against retrieved policies."""
    requirement_id: str
    requirement_text: str
    score: int
    evidence: str
    gap_description: str
    confidence: float
    matching_chunks: List[str]

def assess_coverage(
    req: ExtractedRequirement,
    chunks: List[Tuple[PolicyChunk, float]],
) -> CoverageResult:
    """Assess coverage using LLM or fallback to heuristic."""
    excerpts = "\n\n".join(
        f"[{c.source} — {c.section}] (sim={sim:.3f})\n{c.text}"
        for c, sim in chunks
    )
    prompt = COVERAGE_PROMPT.format(
        obligation_type=req.obligation_type,
        requirement_text=req.text,
        policy_excerpts=excerpts,
    )

    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        # Heuristic fallback: use similarity as proxy
        max_sim = max(sim for _, sim in chunks) if chunks else 0.0
        score = min(5, max(1, int(max_sim * 8)))
        data = {
            "score": score,
            "evidence": chunks[0][0].text[:100] if chunks else "",
            "gap_description": "Mock assessment — set OPENAI_API_KEY for real analysis",
            "confidence": max_sim,
        }

    return CoverageResult(
        requirement_id=req.id,
        requirement_text=req.text,
        score=data["score"],
        evidence=data.get("evidence", ""),
        gap_description=data.get("gap_description", ""),
        confidence=data.get("confidence", 0.5),
        matching_chunks=[c.id for c, _ in chunks],
    )

# ── Demo: assess a few requirements ──
demo_reqs = reqs[:5] if reqs else []
print(f"Assessing {len(demo_reqs)} requirements…\n")
results: List[CoverageResult] = []
for req in demo_reqs:
    top_chunks = retrieve(req.text, top_k=3)
    result = assess_coverage(req, top_chunks)
    results.append(result)
    print(f"  {result.requirement_id:15s}  score={result.score}/5  conf={result.confidence:.2f}")
    print(f"    gap: {result.gap_description[:70]}")

## Section 5 — Gap analysis and remediation

The **Gap Reporter** aggregates coverage results into an actionable report:

| Field | Description |
|-------|-------------|
| `requirement` | The regulation requirement |
| `score` | Coverage score 1-5 |
| `matching_policies` | Policy chunks providing evidence |
| `evidence` | Best supporting quote |
| `gap_description` | What is missing |
| `remediation` | Suggested policy language |
| `priority` | Critical / High / Medium / Low |

In [ ]:
@dataclass
class GapItem:
    """Single gap in compliance coverage."""
    requirement_id: str
    requirement_text: str
    score: int
    matching_policies: List[str]
    evidence: str
    gap_description: str
    remediation: str
    priority: str

def prioritize(score: int, obligation: str) -> str:
    """Assign priority based on score and obligation type."""
    if score <= 1 and obligation == "must":
        return "CRITICAL"
    if score <= 2 and obligation == "must":
        return "HIGH"
    if score <= 3:
        return "MEDIUM"
    return "LOW"

def build_gap_report(
    results: List[CoverageResult],
    requirements: List[ExtractedRequirement],
) -> List[GapItem]:
    """Build a prioritised gap report from coverage results."""
    req_map = {r.id: r for r in requirements}
    gaps: List[GapItem] = []
    for r in results:
        req = req_map.get(r.requirement_id)
        obl = req.obligation_type if req else "must"
        priority = prioritize(r.score, obl)
        remediation = (
            f"Add explicit policy language addressing: {r.gap_description[:80]}"
            if r.score < 4 else "Minor refinement recommended"
        )
        gaps.append(GapItem(
            requirement_id=r.requirement_id,
            requirement_text=r.requirement_text[:80],
            score=r.score,
            matching_policies=r.matching_chunks,
            evidence=r.evidence[:80],
            gap_description=r.gap_description[:80],
            remediation=remediation,
            priority=priority,
        ))
    gaps.sort(key=lambda g: ({"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}[g.priority], g.score))
    return gaps

gap_report = build_gap_report(results, reqs)
print("Gap report (sorted by priority)")
print("=" * 80)
for g in gap_report:
    print(f"\n[{g.priority:8s}] {g.requirement_id} — score {g.score}/5")
    print(f"  Requirement : {g.requirement_text}")
    print(f"  Gap         : {g.gap_description}")
    print(f"  Remediation : {g.remediation}")

# Summary
score_dist = {i: sum(1 for g in gap_report if g.score == i) for i in range(1, 6)}
print(f"\nScore distribution: {score_dist}")
avg_score = np.mean([g.score for g in gap_report]) if gap_report else 0
print(f"Average coverage score: {avg_score:.1f}/5")

## Section 6 — Change monitoring pipeline

The regulation monitor runs periodically to:

1. **Diff** the current regulation text against the last known version
2. **Classify** each change (new requirement, modified threshold, editorial)
3. **Re-assess** only the changed requirements — avoid re-processing everything

In [ ]:
import difflib

@dataclass
class RegulationChange:
    """A detected change in regulation text."""
    article: str
    change_type: str   # new | modified | removed | editorial
    old_text: str
    new_text: str
    impact: str        # HIGH | MEDIUM | LOW
    affected_requirements: List[str]

def detect_changes(
    old_text: str,
    new_text: str,
    article: str = "Unknown",
) -> List[RegulationChange]:
    """Detect and classify changes between regulation versions."""
    changes: List[RegulationChange] = []
    old_lines = old_text.strip().splitlines()
    new_lines = new_text.strip().splitlines()

    matcher = difflib.SequenceMatcher(None, old_lines, new_lines)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            continue
        old_block = "\n".join(old_lines[i1:i2])
        new_block = "\n".join(new_lines[j1:j2])

        if tag == "insert":
            change_type = "new"
            impact = "HIGH"
        elif tag == "delete":
            change_type = "removed"
            impact = "HIGH"
        elif tag == "replace":
            # Check if it's a substantive or editorial change
            if abs(len(old_block) - len(new_block)) < 10:
                change_type = "editorial"
                impact = "LOW"
            else:
                change_type = "modified"
                impact = "MEDIUM"
        else:
            continue

        changes.append(RegulationChange(
            article=article,
            change_type=change_type,
            old_text=old_block[:120],
            new_text=new_block[:120],
            impact=impact,
            affected_requirements=[],
        ))
    return changes

# ── Demo ──
old = """Article 32 — Security of processing
(1) The controller shall implement appropriate technical and organisational measures
to ensure a level of security appropriate to the risk, including:
(a) the pseudonymisation and encryption of personal data;
(b) the ability to ensure confidentiality, integrity, availability and resilience;
(c) the ability to restore access to personal data in a timely manner;
(d) a process for regularly testing and evaluating security measures."""

new = """Article 32 — Security of processing
(1) The controller shall implement appropriate technical and organisational measures
to ensure a level of security appropriate to the risk, including:
(a) the pseudonymisation and encryption of personal data using state-of-the-art algorithms;
(b) the ability to ensure confidentiality, integrity, availability and resilience;
(c) the ability to restore access to personal data within 24 hours of an incident;
(d) a process for regularly testing and evaluating security measures;
(e) automated vulnerability scanning of all systems processing personal data."""

changes = detect_changes(old, new, article="Art. 32")
print(f"Detected {len(changes)} changes in Article 32\n")
for c in changes:
    print(f"  [{c.impact:6s}] {c.change_type:10s}")
    if c.old_text:
        print(f"    OLD: {c.old_text[:80]}")
    if c.new_text:
        print(f"    NEW: {c.new_text[:80]}")
    print()

print("Pipeline: only re-assess requirements affected by MEDIUM+ changes")
re_assess = [c for c in changes if c.impact in ("HIGH", "MEDIUM")]
print(f"  → {len(re_assess)} requirement(s) need re-assessment")

## Exercises

1. **Prompt engineering** — Modify the `COVERAGE_PROMPT` in Section 4 to include
   chain-of-thought reasoning. Add a `reasoning` field to the JSON output. Does this
   improve scoring accuracy on edge cases?

2. **Multi-framework support** — Extend the `ExtractedRequirement` schema with a
   `framework: str` field. Implement extraction for three SOC 2 Trust Service Criteria
   and show that the same policy corpus can be assessed against both GDPR and SOC 2.

3. **Chunking strategies** — Implement an overlapping chunking strategy (50-word
   overlap) in `chunk_policy`. Compare retrieval recall with and without overlap on
   five test queries.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | The pipeline has six stages: extract → retrieve → assess → report → remediate → monitor |
| 2 | Requirement extraction must preserve hierarchy and obligation types |
| 3 | Policy chunking should respect section boundaries and include metadata |
| 4 | Coverage assessment combines retrieval similarity with LLM judgement |
| 5 | Gap reports must be prioritised by score × obligation severity |
| 6 | Change monitoring diffs regulation versions and triggers targeted re-assessment |

## What's Next

In **02 — Build**, we implement the full end-to-end pipeline with a realistic
GDPR regulation corpus, three company policies with intentional gaps, and
OpenAI-powered coverage assessment.

---

## References

1. European Parliament, *Regulation (EU) 2016/679 — General Data Protection Regulation*, 2016.
2. NIST, *AI Risk Management Framework (AI RMF 1.0)*, 2023.
3. ChromaDB documentation — <https://docs.trychroma.com/>
4. Sentence-Transformers library — <https://www.sbert.net/>